In [2]:
from torch.utils.data import Dataset, DataLoader

In [3]:
import pandas as pd

In [4]:
from peft import PeftModel, PeftConfig
from PIL import Image
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from peft import LoraConfig, get_peft_model, TaskType
from transformers.models.blip import modeling_blip_text
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

In [5]:
from transformers import BlipProcessor, BlipForConditionalGeneration

In [6]:
from pycocoevalcap.cider.cider import Cider

In [7]:
import evaluate

In [8]:
from components.dataset import ChestIUDataset
from components.metrics import calculate_bleu, calculate_cider, calculate_meteor, calculate_radgraph_f1
from components.utils import clean_findings

In [9]:
def custom_forward(self, input_ids=None, position_ids=None, inputs_embeds=None, past_key_values_length=0):
    if inputs_embeds is None:
        inputs_embeds = self.word_embeddings(input_ids)
    
    embeddings = inputs_embeds

    if self.position_embedding_type == "absolute":
        if position_ids is None:
            if input_ids is not None:
                seq_length = input_ids.shape[1]
                position_ids = torch.arange(past_key_values_length, seq_length + past_key_values_length, dtype=torch.long, device=embeddings.device)
                position_ids = position_ids.unsqueeze(0).expand(input_ids.shape[:2])
            else:
                position_ids = self.create_position_ids_from_inputs_embeds(inputs_embeds)

        position_embeddings = self.position_embeddings(position_ids)
        embeddings = embeddings + position_embeddings 

    embeddings = self.LayerNorm(embeddings)
    embeddings = self.dropout(embeddings)
    return embeddings

# Apply the patch
modeling_blip_text.BlipTextEmbeddings.forward = custom_forward

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_RUN = "blip-chest-xray-lora-trial-best"

In [19]:
# Hyperparameters
BATCH_SIZE = 16 
EPOCHS = 20  
LR = 1e-4  
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.1
LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = ["query", "value", "key", "dense"]
LORA_DROPOUT = 0.2
MAX_NEW_TOKENS = 200
NUM_BEAMS = 4
REPETITION_PENALTY = 1.3


train_df = pd.read_csv('data/train_data.csv')
val_df = pd.read_csv('data/val_data.csv')

  # Clean the training data
train_df = clean_findings(train_df)
val_df = clean_findings(val_df)


# Load Data
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
train_dataset = ChestIUDataset("data/images/images_normalized", train_df, processor)
val_dataset = ChestIUDataset("data/images/images_normalized", val_df, processor)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [13]:
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model.enable_input_require_grads()

config = LoraConfig(
    r=LORA_R, 
    lora_alpha=LORA_ALPHA, 
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

num_training_steps = EPOCHS * len(train_dataloader)
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

trainable params: 3,858,432 || all params: 227,830,076 || trainable%: 1.6936


In [15]:

model.to(device)

print("Starting Training...")
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
print(f"Total training steps: {num_training_steps}, Warmup steps: {num_warmup_steps}")

best_val_loss = float('inf')
patience_counter = 0
patience = 2  
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1} [Train]")

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)

        labels = input_ids.clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        outputs = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        train_loss += loss.item()
        current_lr = scheduler.get_last_lr()[0]
        progress_bar.set_postfix({"loss": loss.item(), "lr": f"{current_lr:.2e}"})

    avg_train_loss = train_loss / len(train_dataloader)
    train_losses.append(avg_train_loss)

    model.eval()
    val_loss = 0
    val_progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1} [Val]")

    with torch.no_grad():
        for batch in val_progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            labels = input_ids.clone()
            labels[labels == processor.tokenizer.pad_token_id] = -100
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            val_loss += outputs.loss.item()
            val_progress_bar.set_postfix({"val_loss": outputs.loss.item()})

    avg_val_loss = val_loss / len(val_dataloader)
    val_losses.append(avg_val_loss)
    current_lr = scheduler.get_last_lr()[0]

    print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {current_lr:.2e}")
    
    # Early stopping logic
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        print(f"New best validation loss! Saving checkpoint...")
        model.save_pretrained(MODEL_RUN)
    else:
        patience_counter += 1
        print(f"No improvement for {patience_counter} epoch(s)")
        if patience_counter >= patience:
            print(f"Early stopping triggered! Best val loss: {best_val_loss:.4f}")
            break

Starting Training...
Train samples: 2318, Val samples: 498
Total training steps: 2900, Warmup steps: 290


Epoch 1 [Train]:   0%|          | 0/145 [00:00<?, ?it/s]

Epoch 1 [Val]: 100%|██████████| 32/32 [00:56<00:00,  1.77s/it, val_loss=4.47]


Epoch 1 - Train Loss: 6.7165 | Val Loss: 4.4496 | LR: 5.41e-05
New best validation loss! Saving checkpoint...


Epoch 2 [Val]: 100%|██████████| 32/32 [00:54<00:00,  1.71s/it, val_loss=3.03]


...

Epoch 18 - Train Loss: 1.8386 | Val Loss: 2.0634 | LR: 2.77e-06
New best validation loss! Saving checkpoint...


Epoch 19 [Val]: 100%|██████████| 32/32 [00:58<00:00,  1.83s/it, val_loss=1.93]


Epoch 19 - Train Loss: 1.8341 | Val Loss: 2.0617 | LR: 6.39e-07
New best validation loss! Saving checkpoint...


Epoch 20 [Val]: 100%|██████████| 32/32 [00:54<00:00,  1.69s/it, val_loss=1.93]


Epoch 20 - Train Loss: 1.8312 | Val Loss: 2.0614 | LR: 5.22e-09
New best validation loss! Saving checkpoint...


In [16]:
# Evaluate

In [17]:
df_test = pd.read_csv('data/test_data.csv')

In [20]:
base_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = PeftModel.from_pretrained(base_model, MODEL_RUN)
model.to(device)
model.eval()

test_dataset = ChestIUDataset("data/images/images_normalized", df_test, processor)

test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

all_captions = []
with torch.no_grad():
    for batch in tqdm(test_dataloader):
        pixel_values = batch["pixel_values"].to(device)
        
        generated_ids = model.generate(
            pixel_values=pixel_values,
            max_new_tokens=MAX_NEW_TOKENS,        
            num_beams=NUM_BEAMS,
            repetition_penalty=REPETITION_PENALTY,
        )
        
        # Decode all captions in the batch
        captions = processor.batch_decode(generated_ids, skip_special_tokens=True)
        all_captions.extend(captions)

df_test['generated_captions'] = all_captions
print(f"Generated {len(all_captions)} captions")

100%|██████████| 8/8 [02:10<00:00, 16.26s/it]

Generated 491 captions


In [21]:
df_test['generated_captions'].iloc[-5]

'the cardiomediastinal silhouette and pulmonary vasculature are within normal limits in size and contour, but the lungs are clear of any focal airspace disease, pneumothorax, or pleural effusion person is unremarkable with atherosclerotic calcified mediastinal calcified granuloma which may represent a hilar lymph or pleural effusion without acute bony abnormality to suggest a mild degenerative changes of the thoracic spine'

In [22]:
df_test['findings'].iloc[-5]

'Heart size and mediastinal contour are normal. Pulmonary vascularity is normal. Lungs are clear. No pleural effusions or pneumothoraces.'

In [23]:
predictions = df_test['generated_captions'].values

In [24]:
references = df_test['findings'].values

In [25]:
bleu_score = calculate_bleu(generated_captions=predictions, ground_truth_captions=references)
cider_score = calculate_cider(generated_captions=predictions, ground_truth_captions=references)
meteor_score = calculate_meteor(generated_captions=predictions, ground_truth_captions=references)
radgraph_f1_score = calculate_radgraph_f1(generated_captions=predictions, ground_truth_captions=references)

[nltk_data] Downloading package wordnet to /home/user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/user/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/user/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Using device: cuda:0
model_type not provided, defaulting to radgraph-xl


In [26]:
import json
from datetime import datetime

# Collect all training configuration and results
training_config = {
    "experiment_name": "blip-chest-xray-lora-trial",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    
    # Model configuration
    "model": {
        "base_model": "Salesforce/blip-image-captioning-base",
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "trainable_params": 3858432,
        "all_params": 227510588,
        "trainable_percentage": 1.7
    },
    
    # Training hyperparameters
    "training": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "optimizer": "AdamW",
        "scheduler": True,
        "patience": 2,
        "device": "cuda"
    },
    
    # Data information
    "data": {
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
        "test_samples": len(df_test),
        "cleaned_xxxx_tokens": True
    },
    
    # Generation parameters
    "generation": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_beams": NUM_BEAMS,
        "repetition_penalty": REPETITION_PENALTY,
    },
    
    # Training results
    "results": {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "best_val_loss": best_val_loss,
        "final_train_loss": train_losses[-1],
        "final_val_loss": val_losses[-1],
        "epochs_trained": len(train_losses)
    },
    
    # Evaluation metrics
    "evaluation": {
        "bleu_4": bleu_score,
        "cider": cider_score,
        "meteor": meteor_score,
        "radgraph_f1": radgraph_f1_score,
    },
    
    "gpu_profiling": {
        "gpu_device": "NVIDIA H100",
        "gpu_memory": "10885",
        "total_training_time": "107 minutes"
    }
}

# Save to JSON file
output_filename = f"training_results_trial_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_filename, 'w') as f:
    json.dump(training_config, f, indent=2)

In [27]:
df_test['generated_captions'].value_counts()

generated_captions
the cardiomediastinal silhouette and pulmonary vasculature are within normal limits for size and contour of the thoracic aortic, but the lungs are clear of focal airspace disease, pneumothorax, or pleural effusion                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                21
the cardiomediastinal silhouette and pulmonary vasculature are within normal limits for size and contour of the thoracic aortic, without focal airspace opacity, pleural effu